In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/vehicle_data"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/vehicle_data/vehicles.csv,vehicles.csv,1447955215,1785084693000


In [0]:
BRONZE_TABLE = "workspace.default.bronze_vehicles"

csv_path = "/Volumes/workspace/default/vehicle_data/vehicles.csv"

df_raw = (
    spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("multiLine", "true")
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .csv(csv_path)
)

print("Rows:", df_raw.count())

df_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(BRONZE_TABLE)

Rows: 426880


In [0]:
# =============================================================================
# Phase 1 — Spark ETL Pipeline : Bronze → Silver Layer
# Craigslist Used Vehicle Dataset — Data Cleaning
# =============================================================================

In [0]:
import logging
import time
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType
from pyspark.sql import SparkSession, Window   # ← add Window here

In [0]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
log = logging.getLogger("VehicleETL_Silver")

In [0]:
# =============================================================================
# CONFIG
# =============================================================================

In [0]:
BRONZE_TABLE           = "workspace.default.bronze_vehicles"
BRONZE_VALIDATED_TABLE = "workspace.default.vehicles_validated"
SILVER_TABLE           = "workspace.default.silver_vehicles"

In [0]:
CURRENT_YEAR = datetime.now().year + 1

CATEGORY_B_COLS = ["size", "condition", "cylinders", "drive", "paint_color", "type"]

MANUFACTURER_LOOKUP = {
    # Ford
    "f-150": "ford", "f150": "ford", "mustang": "ford", "explorer": "ford",
    "escape": "ford", "focus": "ford", "fusion": "ford", "ranger": "ford",
    "expedition": "ford", "f-250": "ford", "f-350": "ford", "edge": "ford",
    # Chevrolet
    "silverado": "chevrolet", "camaro": "chevrolet", "malibu": "chevrolet",
    "tahoe": "chevrolet", "suburban": "chevrolet", "equinox": "chevrolet",
    "colorado": "chevrolet", "cruze": "chevrolet", "impala": "chevrolet",
    "blazer": "chevrolet", "traverse": "chevrolet",
    # Toyota
    "camry": "toyota", "corolla": "toyota", "rav4": "toyota",
    "tacoma": "toyota", "highlander": "toyota", "4runner": "toyota",
    "tundra": "toyota", "prius": "toyota", "sienna": "toyota",
    # Honda
    "civic": "honda", "accord": "honda", "cr-v": "honda",
    "pilot": "honda", "odyssey": "honda", "fit": "honda", "hrv": "honda",
    # Nissan
    "altima": "nissan", "sentra": "nissan", "maxima": "nissan",
    "rogue": "nissan", "frontier": "nissan", "pathfinder": "nissan", "murano": "nissan",
    # Dodge
    "ram": "dodge", "charger": "dodge", "challenger": "dodge",
    "durango": "dodge", "dart": "dodge", "caravan": "dodge",
    # Jeep
    "wrangler": "jeep", "cherokee": "jeep", "grand cherokee": "jeep",
    "compass": "jeep", "renegade": "jeep",
    # BMW
    "3 series": "bmw", "5 series": "bmw", "x5": "bmw", "x3": "bmw", "x1": "bmw",
    # Mercedes
    "c-class": "mercedes-benz", "e-class": "mercedes-benz", "glc": "mercedes-benz",
    "gle": "mercedes-benz",
    # Hyundai
    "elantra": "hyundai", "sonata": "hyundai", "tucson": "hyundai",
    "santa fe": "hyundai", "accent": "hyundai",
    # GMC
    "sierra": "gmc", "yukon": "gmc", "terrain": "gmc", "acadia": "gmc",
    # Subaru
    "outback": "subaru", "forester": "subaru", "impreza": "subaru",
    "crosstrek": "subaru", "legacy": "subaru",
    # Kia
    "sorento": "kia", "sportage": "kia", "optima": "kia", "soul": "kia",
    # Volkswagen
    "jetta": "volkswagen", "passat": "volkswagen", "tiguan": "volkswagen",
    "golf": "volkswagen", "atlas": "volkswagen",
}

CATEGORICAL_NORMALIZE_COLS = [
    "fuel", "condition", "drive", "transmission", "manufacturer",
    "title_status", "paint_color", "type", "size", "cylinders"
]

In [0]:
# =============================================================================
# 1. SPARK SESSION
# =============================================================================


In [0]:
spark = SparkSession.builder \
    .appName("VehicleMarket_ETL_Silver") \
    .getOrCreate()


pipeline_start = time.time()
log.info("="*65)
log.info("  Vehicle Market ETL Pipeline — Bronze → Silver")
log.info("="*65)

2026-07-26 17:17:07 | INFO | =================================================================
2026-07-26 17:17:07 | INFO |   Vehicle Market ETL Pipeline — Bronze → Silver
2026-07-26 17:17:07 | INFO | =================================================================


In [0]:
# =============================================================================
# 2. LOAD RAW DATA (Bronze Layer)
# =============================================================================


In [0]:
log.info("Loading raw data from Bronze Delta table...")
df_raw = spark.read.table(BRONZE_TABLE)
record_count_bronze = df_raw.count()
col_count_bronze = len(df_raw.columns)
log.info(f"Bronze layer loaded | Records: {record_count_bronze} | Columns: {col_count_bronze}")

2026-07-26 17:17:07 | INFO | Loading raw data from Bronze Delta table...
2026-07-26 17:17:08 | INFO | Bronze layer loaded | Records: 426880 | Columns: 26


In [0]:
# =============================================================================
# 3. PRE-CLEANING DATA PROFILING REPORT
# =============================================================================

In [0]:
log.info("Generating pre-cleaning data profile...")

pre_null_counts = {
    c: df_raw.filter(F.col(c).isNull() | (F.trim(F.col(c)) == "")).count()
    for c in df_raw.columns
}

pre_profile = {
    "distinct_manufacturers" : df_raw.select("manufacturer").distinct().count(),
    "distinct_models"        : df_raw.select("model").distinct().count(),
    "top_fuel_types"         : df_raw.groupBy("fuel").count().orderBy(F.desc("count")).limit(5).collect(),
    "top_states"             : df_raw.groupBy("state").count().orderBy(F.desc("count")).limit(5).collect(),
}

log.info(f"  Distinct Manufacturers : {pre_profile['distinct_manufacturers']}")
log.info(f"  Distinct Models        : {pre_profile['distinct_models']}")
log.info("  Top Fuel Types         : " + str([(r['fuel'], r['count']) for r in pre_profile['top_fuel_types']]))
log.info("  Top States             : " + str([(r['state'], r['count']) for r in pre_profile['top_states']]))


2026-07-26 17:17:09 | INFO | Generating pre-cleaning data profile...
2026-07-26 17:17:29 | INFO |   Distinct Manufacturers : 43
2026-07-26 17:17:29 | INFO |   Distinct Models        : 29669
2026-07-26 17:17:29 | INFO |   Top Fuel Types         : [('gas', 356209), ('other', 30728), ('diesel', 30062), ('hybrid', 5170), (None, 3013)]
2026-07-26 17:17:29 | INFO |   Top States             : [('ca', 50614), ('fl', 28511), ('tx', 22945), ('ny', 19386), ('oh', 17696)]


In [0]:
# =============================================================================
# 4. CATEGORY A — DROP HIGH NULL / LOW VALUE COLUMNS
# =============================================================================

In [0]:
cols_dropped = ["county"]
df = df_raw.drop(*cols_dropped)
log.info(f"[Category A] Dropped columns: {cols_dropped}")

2026-07-26 17:17:30 | INFO | [Category A] Dropped columns: ['county']


In [0]:
# =============================================================================
# 5. CATEGORICAL NORMALIZATION (Lowercase + Trim)
# =============================================================================

In [0]:
log.info("Normalizing categorical columns...")
for col in CATEGORICAL_NORMALIZE_COLS:
    if col in df.columns:
        df = df.withColumn(col, F.lower(F.trim(F.col(col))))

# Standardize known variants
transmission_map = {"auto": "automatic", "manual": "manual", "other": "other"}
df = df.withColumn(
    "transmission",
    F.when(F.col("transmission").isin(list(transmission_map.keys())),
           F.col("transmission")).otherwise(F.col("transmission"))
)

log.info(f"  Normalized: {CATEGORICAL_NORMALIZE_COLS}")

2026-07-26 17:17:31 | INFO | Normalizing categorical columns...
2026-07-26 17:17:32 | INFO |   Normalized: ['fuel', 'condition', 'drive', 'transmission', 'manufacturer', 'title_status', 'paint_color', 'type', 'size', 'cylinders']


In [0]:
# =============================================================================
# 6. CATEGORY B — OPTIONAL ATTRIBUTES → "unknown"
# =============================================================================

In [0]:
for col in CATEGORY_B_COLS:
    df = df.withColumn(
        col,
        F.when(F.col(col).isNull() | (F.trim(F.col(col)) == ""), "unknown")
         .otherwise(F.col(col))
    )
log.info(f"[Category B] Filled 'unknown' for: {CATEGORY_B_COLS}")

2026-07-26 17:17:32 | INFO | [Category B] Filled 'unknown' for: ['size', 'condition', 'cylinders', 'drive', 'paint_color', 'type']


In [0]:
# =============================================================================
# 7. VIN — IDENTIFIER FIELD (preserve existing, no imputation)
# =============================================================================
# VIN nulls are intentionally retained. Used only for dedup validation.

In [0]:
log.info("[VIN] Retained as identifier. No imputation applied.")

2026-07-26 17:17:33 | INFO | [VIN] Retained as identifier. No imputation applied.


In [0]:
# =============================================================================
# 8. CATEGORY C — BUSINESS CRITICAL ATTRIBUTES
# =============================================================================

In [0]:
# --- 8a. Manufacturer Inference via contains() matching ---
log.info("Inferring missing manufacturers from model names...")

df = df.withColumn("model_lower", F.lower(F.trim(F.col("model"))))

inferred_manufacturer = F.lit(None).cast("string")
for model_keyword, mfr in MANUFACTURER_LOOKUP.items():
    inferred_manufacturer = F.when(
        F.col("model_lower").contains(model_keyword),
        F.lit(mfr)
    ).otherwise(inferred_manufacturer)

df = df.withColumn(
    "manufacturer",
    F.when(
        F.col("manufacturer").isNull() | (F.col("manufacturer") == ""),
        inferred_manufacturer
    ).otherwise(F.col("manufacturer"))
).withColumn(
    "manufacturer",
    F.when(F.col("manufacturer").isNull(), "unknown").otherwise(F.col("manufacturer"))
).drop("model_lower")

# --- 8b. Model ---
df = df.withColumn(
    "model",
    F.when(F.col("model").isNull() | (F.trim(F.col("model")) == ""), "unknown model")
     .otherwise(F.lower(F.trim(F.col("model"))))
)

# --- 8c. Year — Drop null records ---
rows_before_year_drop = df.count()
df = df.filter(F.col("year").isNotNull() & (F.trim(F.col("year")) != ""))
rows_after_year_drop = df.count()
log.info(f"[Year] Null records removed: {rows_before_year_drop - rows_after_year_drop}")

# --- 8d. Fuel, Title Status, State → "unknown" ---
for col in ["fuel", "title_status", "state"]:   # ← remove "transmission" from here
    df = df.withColumn(
        col,
        F.when(F.col(col).isNull() | (F.trim(F.col(col)) == ""), "unknown")
         .otherwise(F.col(col))
    )

log.info("[Category C] Business critical columns handled.")


2026-07-26 17:17:34 | INFO | Inferring missing manufacturers from model names...
2026-07-26 17:17:35 | INFO | [Year] Null records removed: 1205
2026-07-26 17:17:35 | INFO | [Category C] Business critical columns handled.


In [0]:
# =============================================================================
# 9. DATA VALIDATION — CLEAN FIELDS BEFORE TYPE CASTING
# =============================================================================

In [0]:
log.info("Running data validation before type casting...")

# --- Price Cleaning ---
# Handles: $5,000 | 5k | 10K | "5000 obo" | "call" | "trade" | "best offer"
df = df.withColumn("price_str", F.lower(F.trim(F.col("price"))))
df = df.withColumn("price_str", F.regexp_replace(F.col("price_str"), r"[\$,\s]", ""))
df = df.withColumn(
    "price_str",
    F.when(F.col("price_str").rlike(r"^\d+k$"),
           (F.regexp_replace(F.col("price_str"), "k", "").cast("int") * 1000).cast("string"))
    .otherwise(F.col("price_str"))
)
# Strip non-numeric suffixes like "obo", "firm"
df = df.withColumn("price_str", F.regexp_extract(F.col("price_str"), r"^(\d+)", 1))
df = df.withColumn(
    "price_clean",
    F.when(
    F.col("price_str").rlike(r"^\d+$") &
    (F.col("price_str").cast("int") >= 100) &
    (F.col("price_str").cast("int") <= 150000),
    F.col("price_str")
).otherwise(None)
).drop("price_str")

# --- Odometer Cleaning ---
# Handles: 120k | 120,000 | "120000 miles" | "high miles"
df = df.withColumn("odo_str", F.lower(F.trim(F.col("odometer"))))
df = df.withColumn("odo_str", F.regexp_replace(F.col("odo_str"), r"[,\s]", ""))
df = df.withColumn("odo_str", F.regexp_replace(F.col("odo_str"), r"miles?", ""))
df = df.withColumn(
    "odo_str",
    F.when(F.col("odo_str").rlike(r"^\d+k$"),
           (F.regexp_replace(F.col("odo_str"), "k", "").cast("int") * 1000).cast("string"))
    .otherwise(F.col("odo_str"))
)
df = df.withColumn("odo_str", F.regexp_extract(F.col("odo_str"), r"^(\d+)", 1))
# df = df.withColumn(
#     "odometer_clean",
    # F.when(F.col("odo_str").rlike(r"^\d+$"), F.col("odo_str")).otherwise(None)
df = df.withColumn(
    "odometer_clean",
    F.when(
        F.col("odo_str").rlike(r"^\d+$") &
        (F.col("odo_str").cast("int") >= 0) &
        (F.col("odo_str").cast("int") <= 400000),
        F.col("odo_str")
    ).otherwise(None)
).drop("odo_str")
# ).drop("odo_str")

# --- Year Cleaning ---
df = df.withColumn(
    "year_clean",
    F.when(F.col("year").rlike(r"^\d{4}$"), F.col("year")).otherwise(None)
)

log.info("  Price, odometer, year validated and cleaned.")

2026-07-26 17:17:36 | INFO | Running data validation before type casting...
2026-07-26 17:17:36 | INFO |   Price, odometer, year validated and cleaned.


In [0]:
# =============================================================================
# 10. TYPE CONVERSION + CONVERSION FAILURE REPORT
# =============================================================================

In [0]:
print(df.columns)

['id', 'url', 'region', 'region_url', 'price', 'year', 'manufacturer', 'model', 'condition', 'cylinders', 'fuel', 'odometer', 'title_status', 'transmission', 'VIN', 'drive', 'size', 'type', 'paint_color', 'image_url', 'description', 'state', 'lat', 'long', 'posting_date', 'price_clean', 'odometer_clean', 'year_clean']


In [0]:
log.info("Casting data types...")

price_total    = df.count()
year_total     = price_total
odo_total      = price_total

df = df \
    .withColumn("price", F.expr("try_cast(price AS INT)")) \
    .withColumn("odometer", F.expr("try_cast(odometer AS INT)"))\
    .withColumn("year", F.expr("try_cast(year AS INT)")) \
    .withColumn("lat", F.expr("try_cast(lat AS DOUBLE)")) \
    .withColumn("long", F.expr("try_cast(long AS DOUBLE)")) \
    .withColumn("posting_date",F.expr("try_to_timestamp(posting_date)")) \
    .drop("price_clean", "odometer_clean", "year_clean")

price_fail    = df.filter(F.col("price").isNull()).count()
odometer_fail = df.filter(F.col("odometer").isNull()).count()
year_fail     = df.filter(F.col("year").isNull()).count()

log.info("  Type Conversion Report:")
log.info(f"    {'Column':<15} {'Total':>8} {'Failed':>8} {'Success%':>10}")
log.info(f"    {'price':<15} {price_total:>8} {price_fail:>8} {((price_total-price_fail)/price_total*100):>9.2f}%")
log.info(f"    {'odometer':<15} {odo_total:>8} {odometer_fail:>8} {((odo_total-odometer_fail)/odo_total*100):>9.2f}%")
log.info(f"    {'year':<15} {year_total:>8} {year_fail:>8} {((year_total-year_fail)/year_total*100):>9.2f}%")

2026-07-26 17:17:37 | INFO | Casting data types...
2026-07-26 17:17:39 | INFO |   Type Conversion Report:
2026-07-26 17:17:39 | INFO |     Column             Total   Failed   Success%
2026-07-26 17:17:39 | INFO |     price             425675        5    100.00%
2026-07-26 17:17:39 | INFO |     odometer          425675     4331     98.98%
2026-07-26 17:17:39 | INFO |     year              425675        0    100.00%


In [0]:
# # Break query plan — accumulated transformations after type casting
# df = df.checkpoint()
# log.info("Checkpoint 2 written — query plan reset after type casting.")

In [0]:
# =============================================================================
# 11. ODOMETER MEDIAN IMPUTATION BY MANUFACTURER
#     (Outliers excluded before computing median)
# =============================================================================

In [0]:
log.info("Imputing missing odometer values using manufacturer group median...")

# Window specs
window_model = Window.partitionBy("manufacturer", "model")
window_manuf = Window.partitionBy("manufacturer")

# Compute layered medians
median_by_model = F.percentile_approx("odometer", 0.5).over(window_model)
median_by_manuf = F.percentile_approx("odometer", 0.5).over(window_manuf)

# Global median — computed as a scalar from non-null values
global_median_val = (
    df.filter(F.col("odometer").isNotNull())
      .select(F.percentile_approx("odometer", 0.5))
      .first()[0]
)

# Cascading coalesce: original → model median → manufacturer median → global
df = df.withColumn(
    "odometer",
    F.coalesce(
        F.col("odometer"),
        median_by_model,
        median_by_manuf,
        F.lit(global_median_val)
    )
)

log.info("  Odometer imputation complete (Model → Manufacturer → Global cascade).")

2026-07-26 17:17:40 | INFO | Imputing missing odometer values using manufacturer group median...
2026-07-26 17:17:41 | INFO |   Odometer imputation complete (Model → Manufacturer → Global cascade).


In [0]:
# =============================================================================
# 11b. TRANSMISSION MODE IMPUTATION BY MANUFACTURER + MODEL
# =============================================================================

log.info("Imputing missing transmission values using Manufacturer + Model mode...")

# Step 1: Treat "unknown" and empty as null so they participate in imputation
df = df.withColumn(
    "transmission_clean",
    F.when(
        F.col("transmission").isNull() | (F.col("transmission") == "unknown"),
        None
    ).otherwise(F.col("transmission"))
)

# Step 2: Count each valid transmission value per (manufacturer, model) group
trans_counts = df.filter(F.col("transmission_clean").isNotNull()) \
    .groupBy("manufacturer", "model", "transmission_clean") \
    .agg(F.count("*").alias("trans_count"))

# Step 3: Rank within each (manufacturer, model) group by frequency descending
#         transmission_clean as tie-breaker ensures determinism
window_rank = Window.partitionBy("manufacturer", "model") \
    .orderBy(F.col("trans_count").desc(), F.col("transmission_clean"))

trans_mode = trans_counts \
    .withColumn("rn", F.row_number().over(window_rank)) \
    .filter(F.col("rn") == 1) \
    .select(
        "manufacturer",
        "model",
        F.col("transmission_clean").alias("mode_transmission")
    )

# Step 4: Join the single definitive mode back to the main dataframe
df = df.join(trans_mode, on=["manufacturer", "model"], how="left")

# Step 5: Fill nulls with the mode; fallback to "automatic" if entire group had no data
df = df.withColumn(
    "transmission",
    F.coalesce(
        F.col("transmission_clean"),
        F.col("mode_transmission"),
        F.lit("automatic")
    )
).drop("transmission_clean", "mode_transmission")

log.info("  Transmission imputation complete (Manufacturer + Model mode → 'automatic' fallback).")

2026-07-26 17:17:41 | INFO | Imputing missing transmission values using Manufacturer + Model mode...
2026-07-26 17:17:41 | INFO |   Transmission imputation complete (Manufacturer + Model mode → 'automatic' fallback).


In [0]:
# =============================================================================
# 12. DUPLICATE REMOVAL
# =============================================================================

In [0]:
log.info("Removing duplicate records...")
count_before_dedup = df.count()

df = df.dropDuplicates(["id"])

df_with_vin    = df.filter(F.col("VIN").isNotNull())
df_without_vin = df.filter(F.col("VIN").isNull())
df_with_vin    = df_with_vin.dropDuplicates(["VIN"])
df             = df_with_vin.union(df_without_vin)

df = df.dropDuplicates(["url"])

count_after_dedup = df.count()
log.info(f"  Duplicates removed: {count_before_dedup - count_after_dedup}")

# Fuzzy Duplicate Report (composite key — report only, no hard drop)
fuzzy_dupes = df.groupBy("manufacturer", "model", "year", "price", "odometer") \
    .count() \
    .filter(F.col("count") > 1) \
    .orderBy(F.desc("count"))

fuzzy_dupe_groups = fuzzy_dupes.count()
log.info(f"  Fuzzy duplicate groups detected (same mfr+model+year+price+odo): {fuzzy_dupe_groups}")
log.info("  Note: Fuzzy duplicates reported only — not dropped (legitimate same-spec listings possible).")


2026-07-26 17:17:42 | INFO | Removing duplicate records...
2026-07-26 17:17:47 | INFO |   Duplicates removed: 147028
2026-07-26 17:17:56 | INFO |   Fuzzy duplicate groups detected (same mfr+model+year+price+odo): 16098
2026-07-26 17:17:56 | INFO |   Note: Fuzzy duplicates reported only — not dropped (legitimate same-spec listings possible).


In [0]:
# =============================================================================
# 13. BUSINESS RULE VALIDATION — STAGE 1
# =============================================================================

In [0]:
display(
    df.select("lat", "long")
      .filter(
          (~F.col("lat").rlike(r"^-?\d+(\.\d+)?$")) &
          F.col("lat").isNotNull()
      )
      .limit(20)
)

lat,long


In [0]:

log.info("Applying business rule filters...")
count_before_rules = df.count()

# Price must exist and be positive
df = df.filter(F.col("price").isNotNull() & (F.col("price") > 0))

# Year range — dynamic upper bound
df = df.filter(
    F.col("year").isNotNull() &
    (F.col("year") >= 1980) &
    (F.col("year") <= CURRENT_YEAR)
)

# Coordinate validation — universal range first
df = df.filter(
    F.col("lat").isNull() | ((F.col("lat") >= -90.0) & (F.col("lat") <= 90.0))
)
df = df.filter(
    F.col("long").isNull() | ((F.col("long") >= -180.0) & (F.col("long") <= 180.0))
)

# US coordinate filter (dataset-specific — separated for reusability)
US_LAT_MIN, US_LAT_MAX   = 24.0, 50.0
US_LONG_MIN, US_LONG_MAX = -125.0, -66.0

df = df.filter(
    F.col("lat").isNull() |
    ((F.col("lat") >= US_LAT_MIN) & (F.col("lat") <= US_LAT_MAX))
)
df = df.filter(
    F.col("long").isNull() |
    ((F.col("long") >= US_LONG_MIN) & (F.col("long") <= US_LONG_MAX))
)

count_after_rules = df.count()
log.info(f"  Records removed by business rules: {count_before_rules - count_after_rules}")

2026-07-26 17:18:00 | INFO | Applying business rule filters...
2026-07-26 17:18:04 | INFO |   Records removed by business rules: 33812


In [0]:
# =============================================================================
# 14. STATISTICAL OUTLIER REMOVAL — STAGE 2 (IQR-based)
# =============================================================================


In [0]:

log.info("Applying IQR-based statistical outlier removal...")

def remove_iqr_outliers(df, col_name, lower_pct=0.005, upper_pct=0.995):
    bounds = df.filter(F.col(col_name).isNotNull()).approxQuantile(
        col_name, [lower_pct, upper_pct], 0.01
    )
    lower, upper = bounds[0], bounds[1]
    log.info(f"  IQR bounds [{col_name}]: lower={lower:.0f} | upper={upper:.0f}")
    return df.filter(
        F.col(col_name).isNull() |
        ((F.col(col_name) >= lower) & (F.col(col_name) <= upper))
    ), lower, upper

count_before_iqr = df.count()
df, price_lower, price_upper       = remove_iqr_outliers(df, "price")
df, odometer_lower, odometer_upper = remove_iqr_outliers(df, "odometer")
count_after_iqr = df.count()

log.info(f"  Records removed by IQR outlier filter: {count_before_iqr - count_after_iqr}")



2026-07-26 17:18:05 | INFO | Applying IQR-based statistical outlier removal...
2026-07-26 17:18:09 | INFO |   IQR bounds [price]: lower=1 | upper=1410065407
2026-07-26 17:18:16 | INFO |   IQR bounds [odometer]: lower=0 | upper=10000000
2026-07-26 17:18:20 | INFO |   Records removed by IQR outlier filter: 37


In [0]:
# =============================================================================
# 15. SAVE BRONZE VALIDATED (Intermediate Layer)
# =============================================================================


In [0]:
log.info(f"Writing Bronze Validated layer to Delta table: {BRONZE_VALIDATED_TABLE}")
df.write.format("delta").mode("overwrite").saveAsTable(BRONZE_VALIDATED_TABLE)
log.info("  Bronze Validated Delta table saved.")

2026-07-26 17:18:21 | INFO | Writing Bronze Validated layer to Delta table: workspace.default.vehicles_validated
2026-07-26 17:18:50 | INFO |   Bronze Validated Delta table saved.


In [0]:
# =============================================================================
# 16. WRITE SILVER LAYER
# =============================================================================

In [0]:
log.info(f"Writing Silver layer to Delta table: {SILVER_TABLE}")
df.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)
log.info("  Silver Delta table saved.")

2026-07-26 17:18:51 | INFO | Writing Silver layer to Delta table: workspace.default.silver_vehicles
2026-07-26 17:19:13 | INFO |   Silver Delta table saved.


In [0]:
# =============================================================================
# 17. SILVER LAYER VALIDATION REPORT
# =============================================================================

In [0]:

record_count_silver = df.count()
pipeline_end        = time.time()
execution_time      = round(pipeline_end - pipeline_start, 2)

log.info("\n" + "="*65)
log.info("           SILVER LAYER VALIDATION REPORT")
log.info("="*65)
log.info(f"  Execution Time          : {execution_time}s")
log.info(f"  Bronze Records          : {record_count_bronze}")
log.info(f"  Silver Records          : {record_count_silver}")
log.info(f"  Total Records Removed   : {record_count_bronze - record_count_silver}")
log.info(f"  Columns Dropped         : {cols_dropped}")
log.info(f"  Columns Imputed         : ['manufacturer', 'model', 'odometer', 'fuel', 'transmission', 'title_status', 'state'] + Category B")
log.info(f"  Duplicates Removed      : {count_before_dedup - count_after_dedup}")
log.info(f"  Fuzzy Duplicate Groups  : {fuzzy_dupe_groups}")
log.info(f"  Business Rule Removed   : {count_before_rules - count_after_rules}")
log.info(f"  IQR Outliers Removed    : {count_before_iqr - count_after_iqr}")
log.info(f"  Price IQR Range         : {price_lower:.0f} — {price_upper:.0f}")
log.info(f"  Odometer IQR Range      : {odometer_lower:.0f} — {odometer_upper:.0f}")
log.info(f"  Year Range Applied      : 1980 — {CURRENT_YEAR}")
log.info("")
log.info("  Type Conversion Failures:")
log.info(f"    price     : {price_fail}")
log.info(f"    odometer  : {odometer_fail}")
log.info(f"    year      : {year_fail}")
log.info("")
log.info("  Null Comparison (Before → After Cleaning):")
log.info(f"  {'Column':<20} {'Before':>10} {'After':>10}")
log.info(f"  {'-'*42}")

post_null_counts = {
    c: df.filter(F.col(c).isNull()).count()
    for c in df.columns
}

for col in df.columns:
    before = pre_null_counts.get(col, 0)
    after  = post_null_counts.get(col, 0)
    if before > 0 or after > 0:
        before_pct = before / record_count_bronze * 100
        after_pct  = after  / record_count_silver * 100
        log.info(f"  {col:<20} {before_pct:>9.2f}% {after_pct:>9.2f}%")

log.info("")
log.info("  Final Schema:")
df.printSchema()

# Read back from Delta table to confirm successful write
df_silver_verify = spark.read.table(SILVER_TABLE)
log.info(f"  Silver Delta Table verified: {df_silver_verify.count()} records written.")
log.info("="*65)
log.info(f"  Silver Delta Table '{SILVER_TABLE}' created successfully.")
log.info("="*65)

2026-07-26 17:19:17 | INFO | 
2026-07-26 17:19:17 | INFO |            SILVER LAYER VALIDATION REPORT
2026-07-26 17:19:17 | INFO | =================================================================
2026-07-26 17:19:17 | INFO |   Execution Time          : 130.67s
2026-07-26 17:19:17 | INFO |   Bronze Records          : 426880
2026-07-26 17:19:17 | INFO |   Silver Records          : 244798
2026-07-26 17:19:17 | INFO |   Total Records Removed   : 182082
2026-07-26 17:19:17 | INFO |   Columns Dropped         : ['county']
2026-07-26 17:19:17 | INFO |   Columns Imputed         : ['manufacturer', 'model', 'odometer', 'fuel', 'transmission', 'title_status', 'state'] + Category B
2026-07-26 17:19:17 | INFO |   Duplicates Removed      : 147028
2026-07-26 17:19:17 | INFO |   Fuzzy Duplicate Groups  : 16098
2026-07-26 17:19:17 | INFO |   Business Rule Removed   : 33812
2026-07-26 17:19:17 | INFO |   IQR Outliers Removed    : 37
2026-07-26 17:19:17 | INFO |   Price IQR Range         : 1 — 1410065407


root
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- region: string (nullable = true)
 |-- region_url: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- condition: string (nullable = true)
 |-- cylinders: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: integer (nullable = false)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = false)
 |-- VIN: string (nullable = true)
 |-- drive: string (nullable = true)
 |-- size: string (nullable = true)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- description: string (nullable = true)
 |-- state: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)
 |-- posting_date: timestamp (nullable = true)



2026-07-26 17:21:11 | INFO |   Silver Delta Table verified: 244772 records written.
2026-07-26 17:21:11 | INFO | =================================================================
2026-07-26 17:21:11 | INFO |   Silver Delta Table 'workspace.default.silver_vehicles' created successfully.
2026-07-26 17:21:11 | INFO | =================================================================


In [0]:
df.show(5)

+------------+--------------+----------+--------------------+-------+--------------------+-----+----+---------+-----------+------+--------+------------+------------+-----------------+-------+---------+-------+-----------+--------------------+--------------------+-----+---------+----------+-------------------+
|manufacturer|         model|        id|                 url| region|          region_url|price|year|condition|  cylinders|  fuel|odometer|title_status|transmission|              VIN|  drive|     size|   type|paint_color|           image_url|         description|state|      lat|      long|       posting_date|
+------------+--------------+----------+--------------------+-------+--------------------+-----+----+---------+-----------+------+--------+------------+------------+-----------------+-------+---------+-------+-----------+--------------------+--------------------+-----+---------+----------+-------------------+
|      toyota|         camry|7308988448|https://abilene.c...|abilen

In [0]:
# ====================================================
# DATA VALIDATION AFTER NEW CSV PARSER
# ====================================================

In [0]:
from pyspark.sql import functions as F

# ===========================
# Dataset Summary
# ===========================
print("="*60)
print("DATASET SUMMARY")
print("="*60)
print(f"Rows    : {df_raw.count()}")
print(f"Columns : {len(df_raw.columns)}")
print(df_raw.printSchema())

DATASET SUMMARY
Rows    : 426880
Columns : 26
root
 |-- id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- region: string (nullable = true)
 |-- region_url: string (nullable = true)
 |-- price: string (nullable = true)
 |-- year: string (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- cylinders: string (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: string (nullable = true)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- VIN: string (nullable = true)
 |-- drive: string (nullable = true)
 |-- size: string (nullable = true)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- description: string (nullable = true)
 |-- county: string (nullable = true)
 |-- state: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- long: string (

In [0]:
categorical_cols = [
    "state",
    "transmission",
    "title_status",
    "fuel",
    "drive",
    "condition",
    "type"
]

for col in categorical_cols:
    print("\n" + "="*70)
    print(f"{col.upper()}")

    print("Distinct Values :", df_raw.select(col).distinct().count())

    df_raw.groupBy(col) \
          .count() \
          .orderBy(F.desc("count")) \
          .show(30, truncate=False)


STATE
Distinct Values : 51
+-----+-----+
|state|count|
+-----+-----+
|ca   |50614|
|fl   |28511|
|tx   |22945|
|ny   |19386|
|oh   |17696|
|or   |17104|
|mi   |16900|
|nc   |15277|
|wa   |13861|
|pa   |13753|
|wi   |11398|
|co   |11088|
|tn   |11066|
|va   |10732|
|il   |10387|
|nj   |9742 |
|id   |8961 |
|az   |8679 |
|ia   |8632 |
|ma   |8174 |
|mn   |7716 |
|ga   |7003 |
|ok   |6792 |
|sc   |6327 |
|mt   |6294 |
|ks   |6209 |
|in   |5704 |
|ct   |5188 |
|al   |4955 |
|md   |4778 |
+-----+-----+
only showing top 30 rows

TRANSMISSION
Distinct Values : 4
+------------+------+
|transmission|count |
+------------+------+
|automatic   |336524|
|other       |62682 |
|manual      |25118 |
|NULL        |2556  |
+------------+------+


TITLE_STATUS
Distinct Values : 7
+------------+------+
|title_status|count |
+------------+------+
|clean       |405117|
|NULL        |8242  |
|rebuilt     |7219  |
|salvage     |3868  |
|lien        |1422  |
|missing     |814   |
|parts only  |198   |
+-----

In [0]:
keywords = [
    "address",
    "finance",
    "country",
    "ford",
    "lincoln",
    "http",
    "www",
    "call",
    "phone"
]

for col in categorical_cols:
    print("\nChecking:", col)

    suspicious = df_raw.filter(
        F.lower(F.col(col)).rlike("|".join(keywords))
    )

    print("Suspicious Rows:", suspicious.count())

    suspicious.select(col).show(10, truncate=False)


Checking: state
Suspicious Rows: 0
+-----+
|state|
+-----+
+-----+


Checking: transmission
Suspicious Rows: 0
+------------+
|transmission|
+------------+
+------------+


Checking: title_status
Suspicious Rows: 0
+------------+
|title_status|
+------------+
+------------+


Checking: fuel
Suspicious Rows: 0
+----+
|fuel|
+----+
+----+


Checking: drive
Suspicious Rows: 0
+-----+
|drive|
+-----+
+-----+


Checking: condition
Suspicious Rows: 0
+---------+
|condition|
+---------+
+---------+


Checking: type
Suspicious Rows: 0
+----+
|type|
+----+
+----+



In [0]:
valid_states = [
    "al","ak","az","ar","ca","co","ct","de","fl","ga",
    "hi","id","il","in","ia","ks","ky","la","me","md",
    "ma","mi","mn","ms","mo","mt","ne","nv","nh","nj",
    "nm","ny","nc","nd","oh","ok","or","pa","ri","sc",
    "sd","tn","tx","ut","vt","va","wa","wv","wi","wy","dc"
]

invalid_states = df_raw.filter(
    (~F.lower(F.col("state")).isin(valid_states))
    & F.col("state").isNotNull()
)

print("Invalid State Rows :", invalid_states.count())

invalid_states.select("state") \
              .distinct() \
              .orderBy("state") \
              .show(100, truncate=False)

Invalid State Rows : 0
+-----+
|state|
+-----+
+-----+



In [0]:
valid_transmission = ["automatic", "manual", "other"]

invalid_trans = df_raw.filter(
    (~F.lower(F.col("transmission")).isin(valid_transmission))
    & F.col("transmission").isNotNull()
)

print("Invalid Transmission Rows :", invalid_trans.count())

invalid_trans.select("transmission") \
             .distinct() \
             .show(100, truncate=False)

Invalid Transmission Rows : 0
+------------+
|transmission|
+------------+
+------------+



In [0]:
valid_title = [
    "clean",
    "rebuilt",
    "salvage",
    "lien",
    "missing",
    "parts only"
]

invalid_title = df_raw.filter(
    (~F.lower(F.col("title_status")).isin(valid_title))
    & F.col("title_status").isNotNull()
)

print("Invalid Title Status Rows :", invalid_title.count())

invalid_title.select("title_status") \
             .distinct() \
             .show(100, truncate=False)

Invalid Title Status Rows : 0
+------------+
|title_status|
+------------+
+------------+



In [0]:
for col in categorical_cols:
    print("\nChecking long values in:", col)

    df_raw.filter(
        F.length(F.col(col)) > 30
    ).select(col).show(20, truncate=False)


Checking long values in: state
+-----+
|state|
+-----+
+-----+


Checking long values in: transmission
+------------+
|transmission|
+------------+
+------------+


Checking long values in: title_status
+------------+
|title_status|
+------------+
+------------+


Checking long values in: fuel
+----+
|fuel|
+----+
+----+


Checking long values in: drive
+-----+
|drive|
+-----+
+-----+


Checking long values in: condition
+---------+
|condition|
+---------+
+---------+


Checking long values in: type
+----+
|type|
+----+
+----+



In [0]:
print("="*60)
print("DUPLICATE CHECK - ALL COLUMNS")
print("="*60)

total_rows = df_raw.count()
unique_rows = df_raw.dropDuplicates().count()

print(f"Total Rows           : {total_rows}")
print(f"Unique Rows          : {unique_rows}")
print(f"Duplicate Rows       : {total_rows - unique_rows}")

DUPLICATE CHECK - ALL COLUMNS
Total Rows           : 426880
Unique Rows          : 426880
Duplicate Rows       : 0


In [0]:
from pyspark.sql import functions as F

print("="*60)
print("DUPLICATE IDs")
print("="*60)

duplicate_ids = (
    df_raw.groupBy("id")
          .count()
          .filter(F.col("count") > 1)
)

print("Duplicate ID Groups :", duplicate_ids.count())

duplicate_ids.orderBy(F.desc("count")).show(20, truncate=False)

DUPLICATE IDs
Duplicate ID Groups : 0
+---+-----+
|id |count|
+---+-----+
+---+-----+



In [0]:
print("="*60)
print("DUPLICATE VINs")
print("="*60)

duplicate_vins = (
    df_raw.filter(F.col("VIN").isNotNull())
          .groupBy("VIN")
          .count()
          .filter(F.col("count") > 1)
)

print("Duplicate VIN Groups :", duplicate_vins.count())

duplicate_vins.orderBy(F.desc("count")).show(20, truncate=False)

DUPLICATE VINs
Duplicate VIN Groups : 40280
+-----------------+-----+
|VIN              |count|
+-----------------+-----+
|1FMJU1JT1HEA52352|261  |
|3C6JR6DT3KG560649|235  |
|1FTER1EH1LLA36301|231  |
|5TFTX4CN3EX042751|227  |
|1GCHTCE37G1186784|214  |
|1GTN1TEH5EZ273019|207  |
|3VWF17AT1FM655022|199  |
|JN1AZ4EH8KM420880|198  |
|1FTMF1CP3GKD62143|195  |
|1GTR1WE07DZ143724|194  |
|1GT22REG1GZ401351|180  |
|1G1FF1R79G0140582|172  |
|WMEEJ3BA2DK636540|168  |
|1GCVKREH6GZ228691|167  |
|2GTV2LECXK1123316|160  |
|JA4AP3AU9LU013694|157  |
|WDDTG5CB9FJ051220|156  |
|1GT220CG8CZ231238|151  |
|JM1NDAC74L0413665|150  |
|3TMLU4EN4CM085701|150  |
+-----------------+-----+
only showing top 20 rows


In [0]:
print("="*60)
print("DUPLICATE URLs")
print("="*60)

duplicate_urls = (
    df_raw.filter(F.col("url").isNotNull())
          .groupBy("url")
          .count()
          .filter(F.col("count") > 1)
)

print("Duplicate URL Groups :", duplicate_urls.count())

duplicate_urls.orderBy(F.desc("count")).show(20, truncate=False)

DUPLICATE URLs
Duplicate URL Groups : 0
+---+-----+
|url|count|
+---+-----+
+---+-----+



In [0]:
print("="*60)
print("BEFORE & AFTER DUPLICATE REMOVAL")
print("="*60)

before = df_raw.count()

after = df_raw.dropDuplicates().count()     # or your duplicate logic

print(f"Rows Before : {before}")
print(f"Rows After  : {after}")
print(f"Removed     : {before - after}")

BEFORE & AFTER DUPLICATE REMOVAL
Rows Before : 426880
Rows After  : 426880
Removed     : 0


In [0]:
print("="*70)
print("CHECKING DUPLICATES USING BUSINESS KEYS")
print("="*70)

business_keys = ["id", "VIN", "url"]

business_duplicates = (
    df_raw.groupBy(business_keys)
          .count()
          .filter(F.col("count") > 1)
)

print("Duplicate Business Records :", business_duplicates.count())

business_duplicates.orderBy(F.desc("count")).show(20, truncate=False)

CHECKING DUPLICATES USING BUSINESS KEYS
Duplicate Business Records : 0
+---+---+---+-----+
|id |VIN|url|count|
+---+---+---+-----+
+---+---+---+-----+



In [0]:
from pyspark.sql import functions as F

print("=" * 70)
print("VIN DUPLICATE ANALYSIS")
print("=" * 70)

# Total records
total_rows = df_raw.count()

# Rows with VIN
rows_with_vin = df_raw.filter(F.col("VIN").isNotNull()).count()

# Rows without VIN
rows_without_vin = df_raw.filter(F.col("VIN").isNull()).count()

# Unique VINs
unique_vins = (
    df_raw.filter(F.col("VIN").isNotNull())
          .select("VIN")
          .distinct()
          .count()
)

# Total rows having duplicate VINs
rows_in_duplicate_vins = (
    df_raw.filter(F.col("VIN").isNotNull())
          .groupBy("VIN")
          .count()
          .filter(F.col("count") > 1)
          .agg(F.sum("count").alias("rows"))
          .collect()[0]["rows"]
)

# Number of VIN groups that are duplicated
duplicate_vin_groups = (
    df_raw.filter(F.col("VIN").isNotNull())
          .groupBy("VIN")
          .count()
          .filter(F.col("count") > 1)
          .count()
)

# Expected rows removed if keeping one row per VIN
expected_removed = rows_with_vin - unique_vins

print(f"Total Rows                    : {total_rows:,}")
print(f"Rows with VIN                 : {rows_with_vin:,}")
print(f"Rows without VIN              : {rows_without_vin:,}")
print(f"Unique VINs                   : {unique_vins:,}")
print(f"Duplicate VIN Groups          : {duplicate_vin_groups:,}")
print(f"Rows belonging to Duplicate VINs : {rows_in_duplicate_vins:,}")
print(f"Expected Rows Removed         : {expected_removed:,}")

VIN DUPLICATE ANALYSIS
Total Rows                    : 426,880
Rows with VIN                 : 265,838
Rows without VIN              : 161,042
Unique VINs                   : 118,264
Duplicate VIN Groups          : 40,280
Rows belonging to Duplicate VINs : 187,854
Expected Rows Removed         : 147,574


In [0]:
from pyspark.sql import functions as F

(
    df_raw.filter(F.col("VIN").isNotNull())
          .groupBy("VIN")
          .count()
          .filter(F.col("count") > 1)
          .orderBy(F.desc("count"))
          .show(20, truncate=False)
)

+-----------------+-----+
|VIN              |count|
+-----------------+-----+
|1FMJU1JT1HEA52352|261  |
|3C6JR6DT3KG560649|235  |
|1FTER1EH1LLA36301|231  |
|5TFTX4CN3EX042751|227  |
|1GCHTCE37G1186784|214  |
|1GTN1TEH5EZ273019|207  |
|3VWF17AT1FM655022|199  |
|JN1AZ4EH8KM420880|198  |
|1FTMF1CP3GKD62143|195  |
|1GTR1WE07DZ143724|194  |
|1GT22REG1GZ401351|180  |
|1G1FF1R79G0140582|172  |
|WMEEJ3BA2DK636540|168  |
|1GCVKREH6GZ228691|167  |
|2GTV2LECXK1123316|160  |
|JA4AP3AU9LU013694|157  |
|WDDTG5CB9FJ051220|156  |
|1GT220CG8CZ231238|151  |
|JM1NDAC74L0413665|150  |
|3TMLU4EN4CM085701|150  |
+-----------------+-----+
only showing top 20 rows
